# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All entities are referenced using their `@id` to ensure compatibility with Croissant's schema.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**


In [1]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [2]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata attributes
metadata = dataset.metadata
# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Review available record sets, fields, and their Croissant `@id`s.

All references below use Croissant schema `@id`.


In [3]:
import pprint

# List available record sets and their IDs
record_sets = dataset.record_sets()
print('Available Record Sets:')
for rs in record_sets:
    print(f"- {rs['@id']} | name: {rs['name']}")
    # Print related fields
    print('  Fields:')
    for field in rs.get('field', []):
        print(f"    - {field['@id']} | name: {field['name']} | dataType: {field.get('dataType', 'unknown')}")
    print()

# Select first record set for demonstration
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"Example records in record set '{example_record_set_id}':")
    for x in dataset.records(record_set=example_record_set_id):
        pprint.pprint(x)
        break  # Print only one example record

## 3. Data Extraction

Load tabular data from every record set into a pandas DataFrame.

All parsing refers to record set and field `@id` attributes.

In [4]:
# List record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Extract a DataFrame for each record set
for rs_id in record_set_ids:
    # Get all records
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Show columns available for the first record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    print("Sample records:")
    display(dataframes[first_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)

Filter numeric records, normalize fields, and group by categorical attributes, referencing each by their `@id`.

Let's select a numeric column and a group column from the first record set.

In [5]:
# Inspect available fields
fields_info = record_sets[0].get('field', [])

# Find numeric-type fields (schema:Integer, schema:Float, etc.)
numeric_fields = [f for f in fields_info if f.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']]
groupable_fields = [f for f in fields_info if f.get('dataType') == 'schema:Text']

# Choose one numeric and one group field
numeric_field_id = numeric_fields[0]['@id'] if numeric_fields else None
group_field_id = groupable_fields[0]['@id'] if groupable_fields else None

df = dataframes[first_record_set_id]

if numeric_field_id and numeric_field_id in df.columns and df[numeric_field_id].dtype in [int, float]:
    threshold = 10

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and its relation to a categorical field, referencing both by `@id`.


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Insufficient fields for visualization.")

## 6. Conclusion

This notebook demonstrated loading, overview, extraction, and basic analysis of the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library.

- All dataset entities, fields, columns, and record sets are referenced via their Croissant schema `@id`, ensuring reproducibility and schema compliance.
- The exploratory steps illustrated basic numeric filtering, normalization, grouping, and visualization of clinical variables.
- For further analysis, the field and entity `@id`s can be consulted directly against the schema as shown in the overview.

Explore further clinical or molecular variables using the same approach, always referencing by their Croissant `@id`.